In [1]:
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
import time
from selenium import webdriver
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.chrome.options import Options
import pandas as pd
import locale
# locale.setlocale(locale.LC_TIME, "pt_BR.utf8")
import datetime as dt
import os
import hashlib
import sqlite3
import re
import requests
from datetime import date

In [2]:
# Headless
chrome_options = Options()
chrome_options.add_argument("--headless")  # Ativa modo headless
chrome_options.add_argument("--disable-gpu")  # Recomendado para Windows
chrome_options.add_argument("--window-size=1920,1080")  # Define tamanho da janela
driver = webdriver.Chrome(options=chrome_options)

# # Normal
# options = uc.ChromeOptions()
# driver = uc.Chrome(options=options)

In [3]:
pasta_banco = 'sqlite'
banco_nome = 'banco_noticias.db'
string_banco = pasta_banco + '/' + banco_nome

# cria_banco(string_banco)

#### Funções Gerais

In [4]:
def get_data(texto_data):
    meses = {
        "janeiro": "January", "fevereiro": "February", "março": "March",
        "abril": "April", "maio": "May", "junho": "June",
        "julho": "July", "agosto": "August", "setembro": "September",
        "outubro": "October", "novembro": "November", "dezembro": "December"
    }
    
    # Substitui o mês em português pelo equivalente em inglês
    for pt, en in meses.items():
        if pt in texto_data:
            texto_data = texto_data.replace(pt, en)
            break
    
    if len(texto_data) == 4:
        return dt.datetime.strptime('1 de January de ' + texto_data, "%d de %B de %Y").date()
    else:
        return dt.datetime.strptime(texto_data, "%d de %B de %Y").date()
  
def corrige_dois_pontos(texto):
    # texto = "Olá mundo!"
    if texto.find(":") == -1:
        pattern = r"(2022|2023|2024|2025|2026)"
        texto = re.sub(pattern, r"\1:", texto, count=1)

    return texto
    
def get_gran_registro_from_texto(texto,sefaz,url=''):
    texto = corrige_dois_pontos(texto)
    registro = {}
    registro['hashid'] = hashlib.sha256(texto.encode()).hexdigest()
    registro['sefaz'] = sefaz
    registro['dtnoticia'] = get_data(texto.split(':')[0])
    registro['dsnoticia'] = get_data(texto.split(':')[0])
    registro['txcompleto'] = texto
    registro['url'] = url
    registro['tsatualizacao'] = dt.datetime.now()

    return registro
    
def get_fcc_registro_from_texto(texto,sefaz,url=''):
    registro = {}
    registro['hashid'] = hashlib.sha256(texto.encode()).hexdigest()
    registro['sefaz'] = sefaz
    registro['txcompleto'] = texto
    registro['url'] = url
    registro['tsatualizacao'] = dt.datetime.now()
    
    return registro

def get_cebraspe_from_texto(texto,sefaz,url=''):
    dtnoticia,dsnoticia = texto.split('\n')
    
    registro = {}
    registro['hashid'] = hashlib.sha256(texto.encode()).hexdigest()
    registro['sefaz'] = sefaz
    registro['dtnoticia'] = dt.datetime.strptime(dtnoticia, "%d/%m/%Y %H:%M")
    registro['dsnoticia'] = dsnoticia
    registro['txcompleto'] = texto
    registro['url'] = url
    registro['tsatualizacao'] = dt.datetime.now()
    
    return registro
    
def get_cebraspe_novos_from_texto(texto,url=''):
   
    registro = {}
    registro['hashid'] = hashlib.sha256(texto.encode()).hexdigest()
    registro['txcompleto'] = texto
    registro['url'] = url
    registro['tsatualizacao'] = dt.datetime.now()
    
    return registro
    
def get_registro_from_texto(texto,sefaz,fonte='gran_noticias',url=''):
    if 'fcc' in fonte:
        return get_fcc_registro_from_texto(texto,sefaz,url=url)
    elif 'cebraspe' == fonte:
        return get_cebraspe_from_texto(texto,sefaz,url=url)
    elif 'cebraspe_novos' == fonte:
        return get_cebraspe_novos_from_texto(texto,url=url)
    else:
        return get_gran_registro_from_texto(texto,sefaz,url=url)

#### Funções Telegram

In [5]:
def tg_envia_mensagem(mensagem):
    token = "8216266402:AAGQC-VACBVgjOrUaBoa9JvNBw8zyBKxysI"
    chat_id = "6024331508"
    # mensagem = "Olá! Esta é uma mensagem automática."

    url = f"https://api.telegram.org/bot{token}/sendMessage"
    
    payload = {
        "chat_id": chat_id,
        "text": mensagem
    }
    
    res = requests.post(url, data=payload)
    
    if res.status_code == 200:
        print("Mensagem enviada com sucesso!")
    else:
        print("Erro ao enviar:", res.text)

def tg_envia_resultados(resultados,fonte):
    for resultado in reversed(resultados):
        # print(resultado['txcompleto'])
        if 'sefaz' in resultado.keys():
            uf = resultado['sefaz'].upper()
        else:
            uf = 'NOVOS'
        
        if resultado['enviada'] == 0:
            texto_mensagem = '''[{}][{}] {} {}
            '''.format(
                uf,
                fonte.upper(),
                resultado['txcompleto'],
                resultado['url'],
                )
            print(texto_mensagem)
            tg_envia_mensagem(texto_mensagem)
            seta_envio(resultado['hashid'],string_banco,fonte=fonte)

#### Funções SQLite

In [6]:
def cria_banco(string_banco):
    # pasta_banco = 'sqlite'
    # banco_nome = 'banco_noticias.db'
    # string_banco = pasta_banco + '/' + banco_nome
    
    # Conecta (ou cria) o banco de dados
    os.makedirs(pasta_banco, exist_ok=True)
    conn = sqlite3.connect(string_banco)
    cursor = conn.cursor()
    
    # Cria a tabela se não existir
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS gran_noticias (
        hashid TEXT PRIMARY KEY,      -- chave primária baseada em hash
        sefaz TEXT NOT NULL,
        dtnoticia TEXT NOT NULL,
        dsnoticia TEXT NOT NULL,
        txcompleto TEXT NOT NULL,
        url TEXT,
        enviada  INTEGER NOT NULL DEFAULT 0,
        tsatualizacao TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );
    """)
    
    # Cria a tabela se não existir
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS fcc_linkarquivo (
        hashid TEXT PRIMARY KEY,      -- chave primária baseada em hash
        sefaz TEXT NOT NULL,
        txcompleto TEXT NOT NULL,
        url TEXT,
        enviada  INTEGER NOT NULL DEFAULT 0,
        tsatualizacao TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );
    """)
    
    # Cria a tabela se não existir
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS fcc_publicacao (
        hashid TEXT PRIMARY KEY,      -- chave primária baseada em hash
        sefaz TEXT NOT NULL,
        txcompleto TEXT NOT NULL,
        url TEXT,
        enviada  INTEGER NOT NULL DEFAULT 0,
        tsatualizacao TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );
    """)
    
    # Cria a tabela se não existir
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS cebraspe (
        hashid TEXT PRIMARY KEY,      -- chave primária baseada em hash
        sefaz TEXT NOT NULL,
        dtnoticia TEXT NOT NULL,
        dsnoticia TEXT NOT NULL,
        txcompleto TEXT NOT NULL,
        url TEXT,
        enviada  INTEGER NOT NULL DEFAULT 0,
        tsatualizacao TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );
    """)

    # Cria a tabela se não existir
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS cebraspe_novos (
        hashid TEXT PRIMARY KEY,      -- chave primária baseada em hash
        txcompleto TEXT NOT NULL,
        url TEXT,
        enviada  INTEGER NOT NULL DEFAULT 0,
        tsatualizacao TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );
    """)
    
    conn.commit()
    conn.close()

def insere_registro(registro,string_banco,fonte='gran_noticias'):
    # Conecta ao banco
    conn = sqlite3.connect(string_banco)
    cursor = conn.cursor()

    if 'fcc' in fonte:
        texto_insert = "INSERT INTO {} (hashid, sefaz, txcompleto, url, tsatualizacao) VALUES (?, ?, ?, ?, ?)".format(fonte)
        tupla_dados = (registro['hashid'], registro['sefaz'], registro['txcompleto'], registro['url'], registro['tsatualizacao'].isoformat())
    elif 'cebraspe_novos' == fonte:
        texto_insert = "INSERT INTO {} (hashid, txcompleto, url, tsatualizacao) VALUES (?, ?, ?, ?)".format(fonte)
        tupla_dados = (registro['hashid'], registro['txcompleto'], registro['url'], registro['tsatualizacao'].isoformat())
    else:
        texto_insert = """
        INSERT INTO {} (hashid, sefaz, dtnoticia, dsnoticia, txcompleto, url, tsatualizacao) VALUES (?, ?, ?, ?, ?, ?, ?)
        """.format(fonte)
        tupla_dados = (registro['hashid'], registro['sefaz'], registro['dtnoticia'].isoformat(), registro['dsnoticia'], registro['txcompleto'],registro['url'], registro['tsatualizacao'].isoformat())
    
    # Inserção segura usando placeholders
    # print(texto_insert, tupla_dados)
    cursor.execute(texto_insert, tupla_dados)
    
    # Salva a transação
    conn.commit()
    
    # Fecha a conexão
    conn.close()
    
    print("Registro inserido com sucesso!")
    
def seta_envio(hashid,string_banco,flag=1,fonte='gran_noticias'):
    # Conecta ao banco
    conn = sqlite3.connect(string_banco)
    cursor = conn.cursor()
    
    texto_update = "UPDATE {} SET enviada = ? WHERE hashid = ? ".format(fonte)
    
    # Inserção segura usando placeholders
    cursor.execute(texto_update, 
                   (flag,hashid))
    
    # Salva a transação
    conn.commit()
    
    # Fecha a conexão
    conn.close()
    
    print("Update com sucesso!")

def seta_tsatualizacao(string_banco,fonte='gran_noticias',sefaz=None,tsatualizacao=dt.datetime.now()):
    # Conecta ao banco
    conn = sqlite3.connect(string_banco)
    cursor = conn.cursor()
    
    texto_update = """UPDATE {} SET tsatualizacao = '{}' where sefaz = '{}'""".format(
        fonte,tsatualizacao.strftime("%Y-%m-%d %H:%M:%S"),sefaz)
    
    # Inserção segura usando placeholders
    cursor.execute(texto_update)
    
    # Salva a transação
    conn.commit()
    
    # Fecha a conexão
    conn.close()
    
    print("Update com sucesso!")
    
def get_registro(hashid,string_banco,fonte='gran_noticias'):
    # Conecta ao banco
    conn = sqlite3.connect(string_banco)
    cursor = conn.cursor()
    
    # Monta a consulta
    query = "SELECT * FROM {} WHERE hashid = '{}'".format(fonte,hashid)
    
    # Executa passando a lista como parâmetros
    cursor.execute(query)
    
    # Busca os resultados
    resultados = cursor.fetchall()
    
    conn.close()

    return resultados
    
def get_ultimas_noticias(dias=5,string_banco='',fonte='gran_noticias',modo_silencioso=False):
    # Conecta ao banco
    conn = sqlite3.connect(string_banco)
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()

    data_limite = (dt.datetime.now() - dt.timedelta(days=dias)).strftime("%Y-%m-%d %H:%M:%S")  # Formato compatível com SQLite
    if ('fcc' in fonte) or ('cebraspe_novos' == fonte):
        coluna = 'tsatualizacao'
    else:
        coluna = 'dtnoticia'
        
    query = "SELECT * FROM {} WHERE {} >= ? order by {} desc".format(fonte,coluna,coluna)
    
    # Consulta registros com data posterior a uma semana atrás
    cursor.execute(query, 
                   (data_limite,))
    
    resultados = cursor.fetchall()
    
    if not modo_silencioso:
        for row in resultados:
            if fonte == 'cebraspe_novos':
                print('CEBRASPE Novos',row["txcompleto"])  # Acessa pelo nome da coluna
            else:
                print(row["sefaz"].upper(),row["txcompleto"])  # Acessa pelo nome da coluna
            
            # print(row["email"])

    conn.close()

    return resultados

#### Funções do Selenium

In [7]:
def get_pagina_noticias(driver,sefaz='df',fonte='gran',modo_silencioso=False):
    fcc = {
        'go':{'url': 'https://www.concursosfcc.com.br/concursos/seego125/index.html',},
        'pi':{'url': 'https://www.concursosfcc.com.br/concursos/sefpi124/index.html',},
        'sp':{'url': 'https://www.concursosfcc.com.br/concursos/fazsp125/index.html',},
        'mt':{'url': 'https://www.concursosfcc.com.br/concursos/sefmt125/index.html',},
        # 'ce':{'url': 'https://www.concursosfcc.com.br/concursos//index.html',},
    }
    cebraspe_novos = {        
        'novos':{'url': 'https://www.cebraspe.org.br/concursos/novos',},
    }
    cebraspe = {        
        'rj':{'url': 'https://www.cebraspe.org.br/concursos/SEFAZ_RJ_25_AUDITOR',},
        'se':{'url': 'https://www.cebraspe.org.br/concursos/SEFAZ_SE_25_AUDITOR',},
        'ce_2021':{'url': 'https://www.cebraspe.org.br/concursos/SEFAZ_CE_21',},
        'df_2020':{'url': 'https://www.cebraspe.org.br/concursos/SEEC_AUDITOR_19',},
        'rn':{'url': 'https://www.cebraspe.org.br/concursos/SEFAZ_RN_25_AUDITOR'},
        'df':{'url': 'https://www.cebraspe.org.br/concursos/SEFAZ_DF_25_AUDITOR'},
        # DF
    }    
    
    if 'fcc' in fonte:
        url = fcc[sefaz]['url']
    elif 'cebraspe' == fonte:
        url = cebraspe[sefaz]['url']
    elif 'cebraspe_novos' == fonte:
        url = cebraspe_novos['novos']['url']
    elif sefaz == 'pa':
        url_base = 'https://blog.grancursosonline.com.br/concurso-sefa-{}/#situacao'
        url = url_base.format(sefaz)
    else:
        url_base = 'https://blog.grancursosonline.com.br/concurso-sefaz-{}/#situacao'
        url = url_base.format(sefaz)
        
    
    # Acessa a página protegida
    driver.get(url)
    
    while True:
        page_state = driver.execute_script('return document.readyState;')
        
        if not modo_silencioso:
            print("Notícias {}: {}".format(sefaz.upper(), page_state))
            
        if page_state == 'complete':
            #break
            return url
            
def busca_noticias(driver,fonte='gran_noticias',lista_sefaz=['df'],modo_silencioso=False):
    for sefaz in lista_sefaz:
        url = get_pagina_noticias(driver,sefaz,fonte=fonte,modo_silencioso=modo_silencioso)
        
        lista_noticias = []
        for elemento_texto in get_lista_elementos(driver,fonte=fonte):  
            registro = get_registro_from_texto(elemento_texto,sefaz,fonte=fonte,url=url)
        
            # Consulta Banco de dados
            # print(registro['hashid'],fonte)
            lista_resultados = get_registro(registro['hashid'],string_banco,fonte=fonte)
            if len(lista_resultados) == 0:
                insere_registro(registro,string_banco,fonte=fonte)
                if not modo_silencioso:
                    print(sefaz,registro['txcompleto'])
            else:
                if not modo_silencioso:
                    print('.',end='.')
                
            lista_noticias.append(registro)
        
        if not modo_silencioso:
            print()

def get_gran_noticias_itens(driver):
    titulo_situacao = driver.find_elements(By.XPATH, '//h2[@id="situacao"]')
    ul_posterior = titulo_situacao[0].find_elements(By.XPATH, 'following-sibling::ul')
    lista_li = ul_posterior[0].find_elements(By.XPATH, './li')
    lista_texto = [elemento.text for elemento in lista_li]
    
    return lista_texto

def get_fcc_linkarquivo_itens(driver):
    div_link_arquivo = driver.find_elements(By.XPATH, '//div[@class="linkArquivo"]')
    lista_div_item = div_link_arquivo[0].find_elements(By.XPATH, './div')
    lista_texto = [elemento.text for elemento in lista_div_item]
    
    return lista_texto
    
def get_fcc_publicacao_itens(driver):
    div_publicacao = driver.find_elements(By.XPATH, '//div[@class="rotuloTopico1"][contains(., "Publicações")]')
    if len(div_publicacao) > 0:
        div_posterior = div_publicacao[0].find_elements(By.XPATH, 'following-sibling::div')
        lista_publicacao = div_posterior[0].text.split("\n")
        lista_texto = [pub.lstrip("- ").strip() for pub in lista_publicacao if pub != '']

        return lista_texto
    else:
        return []

def get_cebraspe_itens(driver):
    lista_titulo = ['Acesso a links','Editais, comunicados e informações','Provas e gabaritos']
    lista_texto = []
    for titulo in lista_titulo:
        div_acesso_a_links = driver.find_elements(By.XPATH, '//h2[contains(., "{}")]/ancestor::div[3]'.format(titulo))
        if len(div_acesso_a_links) > 0:
            lista_li_links = div_acesso_a_links[0].find_elements(By.XPATH, './/li')
            lista_texto = lista_texto + [li.text for li in lista_li_links]
            # lista_texto
        
    return lista_texto
    
def get_cebraspe_novos(driver):
    lista_texto = []
    
    div_novos_concursos = driver.find_elements(By.XPATH, '//h1[contains(., "Novos")]/ancestor::div[3]')
    if len(div_novos_concursos) > 0:
        lista_li_cards = div_novos_concursos[0].find_elements(By.XPATH, './/li')
        lista_texto = lista_texto + [li.text.replace('\nMAIS INFORMAÇÕES','').replace('\n',' - ') for li in lista_li_cards]
        # lista_texto
        
    return lista_texto
    
def get_lista_elementos(driver,fonte='gran_noticias'):
    if fonte == 'fcc_linkarquivo':
        return get_fcc_linkarquivo_itens(driver)
    elif fonte == 'fcc_publicacao':
        return get_fcc_publicacao_itens(driver)
    elif fonte == 'cebraspe_novos':
        return get_cebraspe_novos(driver)
    elif fonte == 'cebraspe':
        return get_cebraspe_itens(driver)
    else:
        return get_gran_noticias_itens(driver)        

# Atualização de notícias

|UF|SP|DF|GO|RN|PA|MT|Total|
|---|:-:|:-:|:-:|:-:|:-:|:-:|:-:|
|Vagas|150+50/285+95|115/265|50/75|50/100|50+100+136/100+200+272|30|**495/855**|
|FCC|X||X|||X|**230/390**|
|CEBRASPE||X||X|||**165/365**|
|VUNESP|||||X||**50/100**|

* 50 SEFAZ/PR;
* 45 SEFAZ/RJ;
* 20 SEFAZ/PI;
* 10 SEFAZ/SE:
* **125 Total**

In [ ]:
while True:
    # Gran Noticias
    busca_noticias(driver,lista_sefaz=['sp','mt','rn','pa','df','go','ce'],fonte='gran_noticias',modo_silencioso=True)
    resultados = get_ultimas_noticias(5,string_banco,fonte='gran_noticias',modo_silencioso=True)    
    tg_envia_resultados(resultados,fonte='gran_noticias')
    
    # FCC Link Arquivo
    busca_noticias(driver,lista_sefaz=['go','pi','sp','mt'],fonte='fcc_linkarquivo',modo_silencioso=True)
    resultados = get_ultimas_noticias(5,string_banco,fonte='fcc_linkarquivo',modo_silencioso=True)    
    tg_envia_resultados(resultados,fonte='fcc_linkarquivo')
    
    # FCC Publicacao
    busca_noticias(driver,lista_sefaz=['go','pi','sp','mt'],fonte='fcc_publicacao',modo_silencioso=True)
    resultados = get_ultimas_noticias(5,string_banco,fonte='fcc_publicacao',modo_silencioso=True)    
    tg_envia_resultados(resultados,fonte='fcc_publicacao')
    
    # Cebraspe
    busca_noticias(driver,lista_sefaz=['rn'],fonte='cebraspe',modo_silencioso=True)
    resultados = get_ultimas_noticias(5,string_banco,fonte='cebraspe',modo_silencioso=True)    
    tg_envia_resultados(resultados,fonte='cebraspe')
    
    try:
        # Cebraspe Novos
        busca_noticias(driver,fonte='cebraspe_novos',modo_silencioso=True)
        resultados = get_ultimas_noticias(5,string_banco,fonte='cebraspe_novos',modo_silencioso=True)
        tg_envia_resultados(resultados,fonte='cebraspe_novos')
    except:
        # Caso der erro
        print('e',end='')
        
    # else:
    # Se não houver erro
    print('.',end='')
    # Espera 30 minutos (30 * 60 segundos)
    time.sleep(30 * 60)
    # break

.

21/11/2025

<div align="center">
    
|UF| Vagas | Salario | Situação |
|-|:-:|-:|:-:|
|SP |200 |39.000| 🔥🔥🔥 |
|DF|265| 37.000| 🔥🔥🔥 |
|MT|30| 41.900| 🔥🔥🔥 |
|RN|100| 36.700| 🔥🔥🔥 |
|GO|75| 32.100| 🔥🔥🔥 |
|PA|286| 45.000| 🔥🔥 |
|BA|200| 33.800| 🔥 |
|MA|10| ❄️+20.000| 🔥 |
|AL|50| ❄️28.500| 🔥 |
|CE|  | 33.000| 🔥 |
|ES|  | ❄️20.600| 🔥 |

</div>

# Noticias do Gran

In [ ]:
fonte='gran_noticias'

In [ ]:
busca_noticias(driver,lista_sefaz=['sp','mt','rn','pa','df','go','ce',],fonte=fonte,modo_silencioso=False)

### Noticias dos ultimos 15 dias

In [ ]:
resultados = get_ultimas_noticias(15,string_banco,fonte=fonte)

### Noticias dos ultimos 5 dias

In [ ]:
resultados = get_ultimas_noticias(5,string_banco,fonte=fonte)

### Enviar Mensagem para Telegram

In [ ]:
tg_envia_resultados(resultados,fonte=fonte)

# Noticias da Banca FCC

In [ ]:
lista_sefaz=['go','pi','sp']

### Link Arquivo

In [ ]:
fonte='fcc_linkarquivo'
busca_noticias(driver,lista_sefaz=lista_sefaz,fonte=fonte,modo_silencioso=False)
# seta_tsatualizacao(string_banco,fonte=fonte,sefaz='pi',tsatualizacao=dt.datetime(2025,8,20))
# seta_tsatualizacao(string_banco,fonte=fonte,sefaz='go',tsatualizacao=dt.datetime(2025,5,30))
resultados = get_ultimas_noticias(5,string_banco,fonte=fonte)
tg_envia_resultados(resultados,fonte=fonte)

### Publicações

In [ ]:
fonte='fcc_publicacao'
busca_noticias(driver,lista_sefaz=lista_sefaz,fonte=fonte,modo_silencioso=False)
# seta_tsatualizacao(string_banco,fonte=fonte,sefaz='pi',tsatualizacao=dt.datetime(2025,8,20))
# seta_tsatualizacao(string_banco,fonte=fonte,sefaz='go',tsatualizacao=dt.datetime(2025,5,30))
resultados = get_ultimas_noticias(5,string_banco,fonte=fonte)
tg_envia_resultados(resultados,fonte=fonte)

# Novos Concursos Cebraspe

In [ ]:
# fonte='cebraspe_novos'
# busca_noticias(driver,fonte=fonte,modo_silencioso=False)
# resultados = get_ultimas_noticias(5,string_banco,fonte=fonte)
# tg_envia_resultados(resultados,fonte=fonte)

# Noticias da Banca Cebraspe

In [ ]:
lista_sefaz=['rn']
fonte='cebraspe'
busca_noticias(driver,lista_sefaz=lista_sefaz,fonte=fonte,modo_silencioso=False)
resultados = get_ultimas_noticias(5,string_banco,fonte=fonte)
tg_envia_resultados(resultados,fonte=fonte)

# Liga atualização


# Testes do Telegram


In [ ]:
url = f"https://api.telegram.org/bot{token}/getUpdates"

res = requests.get(url)
if res.status_code == 200:
    updates = res.json()
    print(updates)
else:
    print("Erro ao obter updates:", res.text)

In [ ]:

for update in updates["result"]:
    if "message" in update and "text" in update["message"]:
        chat_id = update["message"]["chat"]["id"]
        texto = update["message"]["text"]
        print(f"Mensagem de {chat_id}: {texto}")

In [ ]:
# {
#     'update_id': 39541705,
#     'message': {
#         'message_id': 2,
#         'from': {'id': 6024331508,'is_bot': False,'first_name': 'Carlos','last_name': 'Freitas','language_code': 'pt-br'},
#         'chat': {'id': 6024331508,'first_name': 'Carlos','last_name': 'Freitas','type': 'private'},
#         'date': 1755959115,
#         'text': 'Oi'
#     }
# }